# ImmunoStruct: Run inference on new data

This notebook runs **ImmunoStruct** immunogenicity prediction on a single peptide-MHC pair.  

You provide the **MHC allele** and **peptide sequence**; we run remote AlphaFold2 folding and then inference.

**Setup:** 

Use the ImmunoStruct conda environment (see README).

Run this notebook from the repo root or from `demo/` so that `immunostruct` is on the path.

A GPU is optional but speeds up structure prediction.

In [ ]:
!export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH

First download AlphaFold weights (likely default to `~/.cache/colabfold`).

In [ ]:
!python -m colabfold.download

Setup: set project root and ensure `immunostruct` is importable. Run this cell first.

In [ ]:
import os
import sys
import re
from pathlib import Path

# Project root: run this notebook from repo root (ImmunoStruct/) or from demo/
cwd = os.getcwd()
if os.path.basename(cwd) == "demo" and os.path.isdir(os.path.join(cwd, "..", "immunostruct")):
    PROJECT_ROOT = os.path.abspath(os.path.join(cwd, ".."))
else:
    PROJECT_ROOT = cwd

os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
_immuno = os.path.join(PROJECT_ROOT, "immunostruct")
if os.path.isdir(_immuno) and _immuno not in sys.path:
    sys.path.insert(0, _immuno)

# Use local immunostruct/preprocessing/colabfold.py, pairmsa.py, colabfold_alphafold.py
_prep = os.path.join(PROJECT_ROOT, "immunostruct", "preprocessing")
if _prep not in sys.path:
    sys.path.insert(0, _prep)
for _m in ("colabfold", "pairmsa", "colabfold_alphafold"):
    if _m in sys.modules:
        del sys.modules[_m]
import colabfold_alphafold as cf_af

print("Project root:", PROJECT_ROOT)
print("Setup done.")

Project root: /gpfs/radev/project/krishnaswamy_smita/cl2482/ImmunoStruct
Setup done.


Download model weights from Hugging Face.

In [ ]:
from huggingface_hub import hf_hub_download
import os

model_weight_dir = "pretrained_models"

REPO_ID = "ChenLiu1996/ImmunoStruct"
REPO_TYPE = "model"

os.makedirs(model_weight_dir, exist_ok=True)
iedb_path = hf_hub_download(repo_id=REPO_ID, filename="IEDB_model_seed1.pt", repo_type=REPO_TYPE, local_dir=model_weight_dir)
cedar_path = hf_hub_download(repo_id=REPO_ID, filename="CEDAR_model_seed2.pt", repo_type=REPO_TYPE, local_dir=model_weight_dir)
print("IEDB model:", iedb_path)
print("CEDAR model:", cedar_path)

IEDB model: pretrained_models/IEDB_model_seed1.pt
CEDAR model: pretrained_models/CEDAR_model_seed2.pt


Provide peptide-MHC data to run inference on.

In [ ]:
peptide = "YTDQISKYA"
allele = "HLA-A*01:01"
# See `data/HLA_allele_sequences.csv` for the allele-sequence mapping for some common MHCs.
MHC_sequence = "SHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQKMEPRAPWIEQEGPEYWDQETRNMKAHSQTDRANLGTLRGYYNQSEDGSHTIQIMYGCDVGPDGRFLRGYRQDAYDGKDYIALNEDLRSWTAADMAAQITKRKWEAVHAAEQRRVYLEGRCVDGLRRYLENGKETLQRTDPPKTHMTHHPISDHEATLRCWALGFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGEEQRYTCHVQHEGLPKPLTLR"

**Step 1. MSA generation + AlphaFold2 folding**

Run MSA using a remote MMseqs2 server.

Run AlphaFold2 to predict the peptide-MHC structure.

In [ ]:
PARAMS_LOC = "/gpfs/radev/home/cl2482/.cache/colabfold"

# --- 1) MSA: prepare inputs and run remote MSA ---
full_sequence_folding = f"{MHC_sequence}:{peptide}"
full_sequence_name = f"{allele}:{peptide}"
full_sequence_name = re.sub(r'\W+', '_', full_sequence_name)

output_dir = os.path.join(PROJECT_ROOT, "tmp_data", "structure", full_sequence_name)
os.makedirs(output_dir, exist_ok=True)

I = cf_af.prep_inputs(full_sequence_folding, full_sequence_name, "1:1", output_dir=output_dir, clean=False)
print(f"Prepared inputs for {full_sequence_name}.", flush=True)

cf_af.prep_msa(
    I,
    msa_method="mmseqs2",
    add_custom_msa=False,
    msa_format="fas",
    pair_mode="unpaired",
    pair_cov=50,
    pair_qid=20,
    TMP_DIR="/tmp/",
)
print(f"Prepared MSA for {full_sequence_name}.", flush=True)

# --- 2) Folding: build features and run AlphaFold2 ---
feature_dict = cf_af.prep_feats(I, clean=False)
opt = {
    "N": len(feature_dict["msa"]),
    "L": len(feature_dict["residue_index"]),
    "use_ptm": True,
    "use_turbo": True,
    "num_relax": None,
    "max_recycles": 3,
    "tol": 0.0,
    "num_ensemble": 1,
    "max_msa_clusters": 256,
    "max_extra_msa": 512,
    "is_training": False,
}

runner = cf_af.prep_model_runner(opt, params_loc=PARAMS_LOC)
_, _ = cf_af.run_alphafold(
    feature_dict, opt, runner,
    num_models=1, num_samples=1, subsample_msa=True,
    rank_by="pLDDT", show_images=False, params_loc=PARAMS_LOC,
)
print(f"Step 1 done. Structure(s) in: {I['output_dir']}", flush=True)

homooligomer: 1:1
total_length: 281
output_dir: /gpfs/radev/project/krishnaswamy_smita/cl2482/ImmunoStruct/tmp_data/structure/HLA_A_01_01_YTDQISKYA
Prepared inputs for HLA_A_01_01_YTDQISKYA.
running mmseqs2
Prepared MSA for HLA_A_01_01_YTDQISKYA.


Running model_1_ptm_seed_0:   0%|          | 0/1 [elapsed: 00:03 remaining: ?]

: 

**Step 2. Preparing inputs to the ImmunoStruct model**

Convert the predicted PDB to a structure graph (MHC + peptide).  

Encode the full sequence and build a single-sample batch.  

In [ ]:
from glob import glob
import os
import numpy as np
import torch
import dgl

from preprocessing.step6_pdb_to_pyg import pdb_to_pyg
from graphein.protein.graphs import read_pdb_to_dataframe
from data_loading.data_utils import pad_graph, to_dgl, pad_peptide_sequence, one_hot_encode_sequence
from data_loading.constants import AMINO_ACIDS, PADDING_CHAR
from models.mapping import model_map

# --- 1) Find PDB from Step 1 ---
pdb_dir = I["output_dir"]
pdb_files = sorted(glob(os.path.join(pdb_dir, "*unrelaxed*.pdb")))
if not pdb_files:
    pdb_files = sorted(glob(os.path.join(pdb_dir, "*.pdb")))
if not pdb_files:
    raise FileNotFoundError(f"No PDB found in {pdb_dir}. Run Step 1 first.")
pdb_path = pdb_files[0]
print(f"Using PDB: {pdb_path}")

# --- 2) Convert PDB to PyG graph ---
g_pyg = pdb_to_pyg(pdb_path)
g_pyg.name = f"{allele}_{peptide}"

# Extract CA coordinates from the PDB for spatial features.
tdf = read_pdb_to_dataframe(pdb_path)
tdfa = tdf[tdf["chain_id"] == "A"].drop_duplicates("residue_number").head(179)
tdfb = tdf[tdf["chain_id"] == "B"].drop_duplicates("residue_number")
coord_cols = ["x_coord", "y_coord", "z_coord"] if "x_coord" in tdf.columns else ["x", "y", "z"]
coords = np.vstack([tdfa[coord_cols].values, tdfb[coord_cols].values]).astype(np.float32)
g_pyg.coords = torch.tensor(coords, dtype=torch.float32)

# --- 3) Prepare features: drop hbond dims, add coords, pad, convert to DGL ---
FEATURE_SIZE, COORD_SIZE = 23, 3
g_pyg.x = torch.cat([g_pyg.x[:, :-2], g_pyg.coords], dim=-1)
g_pyg.y = torch.tensor([0.0, 0.0], dtype=torch.float32)
g_pyg = pad_graph(g_pyg, g_pyg.num_nodes, FEATURE_SIZE, COORD_SIZE)
dgl_g = to_dgl(g_pyg)
batched_graph = dgl.batch([dgl_g])

# --- 4) Encode full sequence (MHC + peptide) ---
full_seq = MHC_sequence + peptide
padded_full = pad_peptide_sequence(full_seq, 283, PADDING_CHAR)
encoded_full = one_hot_encode_sequence(padded_full, AMINO_ACIDS, PADDING_CHAR)
sequence_data = torch.tensor(encoded_full, dtype=torch.float32).unsqueeze(0)
# Use 0.4 as the placeholders for biochemical properties.
peptide_property = torch.tensor([[0.4, 0.4]], dtype=torch.float32)

OSError: libcusparse.so.11: cannot open shared object file: No such file or directory

**Step 3. Run inference on the ImmunoStruct model.**

Load the IEDB ImmunoStruct model and run forward pass.  

Output: predicted immunogenicity score.

In [ ]:
# Pick which model to use.
WHICH_MODEL = "IEDB"
# WHICH_MODEL = "CEDAR"


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = 283 * 21

if WHICH_MODEL == "IEDB":
    model = model_map["HybridModelv2"](vae_input_dim=input_dim, device=device)
    model.load_trained(iedb_path, new_head=False, map_location=device)
elif WHICH_MODEL == "CEDAR":
    model = model_map["HybridModel"](vae_input_dim=input_dim, device=device)
    model.load_trained(cedar_path, new_head=False, map_location=device)
model.to(device)
model.eval()

batched_graph = batched_graph.to(device)
sequence_data = sequence_data.to(device)
peptide_property = peptide_property.to(device)

with torch.no_grad():
    _, _, _, final_output = model(batched_graph, sequence_data, peptide_property)
    prob = torch.sigmoid(final_output).squeeze().item()

print(f"Predicted immunogenicity probability: {prob:.4f}")